# Kuroshio R5.5 — 真实轴线交点保真与单值化失真诊断（不重训）

**核心问题：** R5.4 在 1,275 条路径中观察到第二分支多交点位置约 8.37%，而冻结训练集由“经度排序→同经度平均→线性插值/外推”产生。本实验量化第一、第二分支旧表示点**距真实轴线交点的最小偏差**、多值间距、曲线连续性，并对比三个不依赖模型误差的候选交点策略（首次穿越／最后穿越／沿路径弧长连续动态规划）。

**重要边界：** 不修改原 `final/dataset`、不改任何冻结轨迹、不删除异常日期、不训练/选模型、不用 2008/2016 研发集误差来优化表示策略；与现有 R5.3 的误差分析仅做描述性关联，不得冒充真正独立的物理真值或盲测。候选新表示是否改善预测，必须在后续相同架构、相同起报样本、严格时间分割下公平比较。

**输入：** `/mnt/d/KuroshioDaily1993_2025/final/dataset/daily_dual_branch_paths_1993_2025_FINAL.npz`、`final/tracks_final/`、之前 R5.4 ZIP。

**输出：** 自动保存逐日/逐年几何指标、部分候选表示、关联统计与中英文图（若中文字体可用）、完整 ZIP/SHA256。**即使部分路径读取失败，也归档成功处理的结果。**

In [ ]:
# ===== 0. 配置与依赖 =====
from pathlib import Path
from dataclasses import dataclass, asdict
from IPython.display import display, FileLink, Markdown
import pandas as pd, numpy as np, math, json, time, hashlib, zipfile, io, warnings, sys
import matplotlib
matplotlib.use('Agg')  # VS Code WSL 环境可离屏绘图，不需要 GUI backend
import matplotlib.pyplot as plt
from matplotlib import font_manager

@dataclass
class CFG:
    ROOT: str = '/mnt/d/KuroshioDaily1993_2025'
    FROZEN_NPZ: str = 'final/dataset/daily_dual_branch_paths_1993_2025_FINAL.npz'
    TRACKS: str = 'final/tracks_final'
    R54_ZIP: str = 'R54_上传给ChatGPT_路径表示与误差归因审计.zip'
    OUT_DIR: str = 'R5_5_流轴几何保真与交点策略审计'
    # 默认完整审计 2008/2016 两个既往研发年份，2012/2020 为额外研发确认，
    # 并从其余年份按时间均匀抽样。FULL_SCAN=True 可以审计全部 11,771 天。
    FOCUS_YEARS: tuple = (2008, 2012, 2016, 2020)
    SAMPLES_PER_OTHER_YEAR: int = 24
    FULL_SCAN: bool = False
    RESUME: bool = True
    SAVE_EVERY: int = 50
    FIG_DPI: int = 300
    MAX_ARCHIVE_MB: int = 120

cfg=CFG()
ROOT=Path(cfg.ROOT); NPZ=ROOT/cfg.FROZEN_NPZ; TRACKS=ROOT/cfg.TRACKS
OUT=ROOT/cfg.OUT_DIR; DATA=OUT/'实验数据'; REPORT=OUT/'报告'; IM_ZH=OUT/'图表'/'中文'; IM_EN=OUT/'图表'/'英文'
for d in [DATA,REPORT,IM_ZH,IM_EN]:d.mkdir(parents=True,exist_ok=True)
print('Python',sys.executable,'NPZ',NPZ.exists(),'tracks dir',TRACKS.exists(),'R54 ZIP',(ROOT/cfg.R54_ZIP).exists())
if not NPZ.exists() or not TRACKS.exists():raise FileNotFoundError('缺少冻结 NPZ 或 final/tracks_final，请核对 cfg.ROOT。')
with np.load(NPZ,allow_pickle=False) as z:
    dates=pd.to_datetime(z['dates']).normalize()
    # B2 输入与原训练的经度范围保持一致
    all_g=np.asarray(z['branch2_target_lons'],dtype=float)
    mask_g=all_g<=145.0+1e-8
    GRID=all_g[mask_g].copy()
    FROZEN_B2=np.asarray(z['branch2_lat'],dtype=np.float32)[:,mask_g]
    GRID_B1=np.asarray(z['branch1_target_lats'],float)[12:]
    FROZEN_B1=np.asarray(z['branch1_lon'],dtype=np.float32)[:,12:]
assert FROZEN_B1.shape==(len(dates),len(GRID_B1))
assert FROZEN_B2.shape==(len(dates),len(GRID))
assert np.all(np.diff(GRID)>0), 'B2 经度网格必须严格递增'
DATE_ROW={pd.Timestamp(dt).normalize():i for i,dt in enumerate(dates)}
print('Frozen days',len(dates),'B2 longitude samples',len(GRID),'range',GRID.min(),GRID.max())

In [ ]:
# ===== 1. 多交点提取 + 沿路径弧长一致的动态规划（不使用预测误差） =====
def haversine_segment_km(lon,lat):
    x1=np.radians(np.asarray(lon[:-1],float)); x2=np.radians(np.asarray(lon[1:],float))
    y1=np.radians(np.asarray(lat[:-1],float)); y2=np.radians(np.asarray(lat[1:],float))
    a=np.sin((y2-y1)/2)**2 + np.cos(y1)*np.cos(y2)*np.sin((x2-x1)/2)**2
    return 6371.0*2*np.arctan2(np.sqrt(np.clip(a,0,1)),np.sqrt(np.maximum(0,1-a)))

def all_crossings(lon,lat,grids):
    """每条固定经线与有向原始流轴的交点列表 (latitude, alongpath_km)。
    半开线段消除大部分共享端点重复；经度恒定线段不进入分母。
    """
    lon=np.asarray(lon,dtype=float);lat=np.asarray(lat,dtype=float);grids=np.asarray(grids,float)
    mask=np.isfinite(lon)&np.isfinite(lat)
    lon=lon[mask];lat=lat[mask]
    if len(lon)<3:raise ValueError('有效轨迹点数不足3')
    arclen=np.r_[0.,np.cumsum(haversine_segment_km(lon,lat))]
    x0,x1=lon[:-1],lon[1:]; y0,y1=lat[:-1],lat[1:]
    good=np.isfinite(x0)&np.isfinite(x1)&np.isfinite(y0)&np.isfinite(y1)&(np.abs(x1-x0)>1e-10)
    res=[]
    for g in grids:
        idx=np.flatnonzero(good&(((x0<=g)&(x1>g))|((x1<=g)&(x0>g))))
        if idx.size==0:
            res.append(np.empty((0,2),float));continue
        t=(g-x0[idx])/(x1[idx]-x0[idx])
        yi=y0[idx]+t*(y1[idx]-y0[idx])
        si=arclen[idx]+t*(arclen[idx+1]-arclen[idx])
        order=np.argsort(si)
        res.append(np.column_stack([yi,si])[order])
    return res

def latitude_crossings(lon,lat,grids):
    """B1 固定纬线的经度交点及真实地理弧长（不交换经纬度计算距离）。"""
    lon=np.asarray(lon,float);lat=np.asarray(lat,float);grids=np.asarray(grids,float)
    good=np.isfinite(lon)&np.isfinite(lat);lon=lon[good];lat=lat[good]
    if len(lat)<3:raise ValueError('有效流轴点不足3')
    arclen=np.r_[0.,np.cumsum(haversine_segment_km(lon,lat))]
    a=lat[:-1];b=lat[1:]; x=lon[:-1];y=lon[1:]
    res=[]
    for g in grids:
        idx=np.flatnonzero((((a<=g)&(b>g))|((b<=g)&(a>g))) & (np.abs(b-a)>1e-10))
        if len(idx)==0:res.append(np.empty((0,2),float));continue
        t=(g-a[idx])/(b[idx]-a[idx])
        coords=x[idx]+t*(y[idx]-x[idx])
        ss=arclen[idx]+t*(arclen[idx+1]-arclen[idx])
        order=np.argsort(ss)
        res.append(np.column_stack([coords,ss])[order])
    return res

def branch1_scores(lon,lat,lats_grid,old_lon):
    xs=latitude_crossings(lon,lat,lats_grid)
    counts=np.array([len(a) for a in xs])
    scales=111.2*np.cos(np.radians(lats_grid))
    nearest=np.array([np.min(np.abs(a[:,0]-v))*scale if len(a) and np.isfinite(v) else np.nan
                      for a,v,scale in zip(xs,old_lon,scales)],float)
    spans=np.array([np.ptp(a[:,0])*scale if len(a)>1 else np.nan for a,scale in zip(xs,scales)],float)
    ok=np.isfinite(nearest)
    return {'第一分支无交点比例_复核':float(np.mean(counts==0)),
            '第一分支多交点比例_复核':float(np.mean(counts>1)),
            '第一分支多交点纬线经度跨度均值_km':float(np.nanmean(spans)) if np.isfinite(spans).any() else 0.0,
            '第一分支旧表示距真实交点平均_km':float(np.nanmean(nearest)) if ok.any() else np.nan,
            '第一分支旧表示距真实交点P95_km':float(np.nanpercentile(nearest,95)) if ok.any() else np.nan}

def continuous_arc_selector(intersections, grid):
    """对有交点的相邻经线块进行单调弧长 DP。
    只解决一个**单值分支**的选择，不可恢复未被选择的合法蛇行分支。
    不允许后一经线沿原始流轴向后移动（s 减小）。
    """
    n=len(grid); selected=np.full(n,np.nan); picked_s=np.full(n,np.nan)
    costs=[];prev=[]; restart=0
    # 缺口中断一段 DP；不强行外推
    for i in range(n):
        arr=intersections[i]
        if len(arr)==0:
            costs.append(None);prev.append(None);continue
        if i==0 or len(intersections[i-1])==0:
            costs.append(np.zeros(len(arr),float)+0.001*arr[:,1]);prev.append(np.full(len(arr),-1,int))
            continue
        prev_arr=intersections[i-1];p_cost=costs[i-1]
        cost=np.full(len(arr),np.inf);link=np.full(len(arr),-1,int)
        for j,(y,s) in enumerate(arr):
            ds=s-prev_arr[:,1]
            ok=ds>=-0.001
            if not ok.any():continue
            # 左端变化只影响平滑性；过大的弧长跃迁代价更高，但不是硬性剪枝
            dy=111.2*np.abs(y-prev_arr[:,0])
            transition=dy + 0.07*np.maximum(ds,0) + 0.01*np.maximum(ds-120,0)
            cand=p_cost+transition
            cand=np.where(ok,cand,np.inf)
            k=int(np.argmin(cand))
            if np.isfinite(cand[k]):cost[j]=cand[k];link[j]=k
        if not np.isfinite(cost).any():
            restart+=1
            cost=np.zeros(len(arr),float)+0.001*arr[:,1]
            link[:]=-1
        costs.append(cost);prev.append(link)
    # 回溯每个连续子段，并允许不可达时被动拆段
    j=n-1
    while j>=0:
        if not len(intersections[j]):j-=1;continue
        c=costs[j];k=int(np.argmin(c))
        while j>=0 and len(intersections[j]):
            if not np.isfinite(c[k]):break
            selected[j]=intersections[j][k,0];picked_s[j]=intersections[j][k,1]
            parent=int(prev[j][k]);j-=1
            if parent<0:break
            if j<0 or len(intersections[j])==0:break
            k=parent;c=costs[j]
    return selected,picked_s,restart

def grid_reconstruction_scores(lon,lat,grid,old_b2):
    xs=all_crossings(lon,lat,grid)
    counts=np.array([len(q) for q in xs],int)
    first=np.array([q[0,0] if len(q) else np.nan for q in xs]);last=np.array([q[-1,0] if len(q) else np.nan for q in xs])
    dp,ps,restart=continuous_arc_selector(xs,grid)
    nearest=np.array([float(np.min(np.abs(q[:,0]-v))*111.2) if len(q) and np.isfinite(v) else np.nan for q,v in zip(xs,old_b2)])
    spans=np.array([float(np.ptp(q[:,0])*111.2) if len(q)>=2 else np.nan for q in xs])
    spread=spans[np.isfinite(spans)]
    # 候选纬度属于真实线段交点，因此局地 off-axis 距离近零；
    # 这不能作为模型预测改善的证据，需再量化跳跃、不唯一性和分支歧义。
    def jump95(x):
        z=np.abs(np.diff(x))*111.2
        z=z[np.isfinite(z)]
        return float(np.percentile(z,95)) if len(z) else np.nan
    old_near=nearest[np.isfinite(nearest)]
    old_multi=np.array([v for v,k in zip(nearest,counts) if k>1 and np.isfinite(v)])
    valid=counts>0
    metrics={
        '经线格点数':len(grid),'无交点比例':float(np.mean(counts==0)),
        '多交点比例':float(np.mean(counts>=2)),
        '多交点平均跨度_km':float(np.mean(spread)) if len(spread) else 0.,
        '多交点跨度P95_km':float(np.percentile(spread,95)) if len(spread) else 0.,
        '旧表示距任一真实交点平均_km':float(np.mean(old_near)) if len(old_near) else np.nan,
        '旧表示距任一真实交点P95_km':float(np.percentile(old_near,95)) if len(old_near) else np.nan,
        '旧表示偏离真实交点超过5km比例':float(np.mean(nearest[valid]>5.0)) if valid.any() else np.nan,
        '旧表示多交点位置偏差均值_km':float(np.mean(old_multi)) if len(old_multi) else np.nan,
        '旧表示沿经度逐格纬度跳跃P95_km':jump95(old_b2),
        '首次交点逐格纬度跳跃P95_km':jump95(first),
        '末次交点逐格纬度跳跃P95_km':jump95(last),
        '连续弧长DP逐格纬度跳跃P95_km':jump95(dp),
        '连续弧长DP重启段数':restart,
        '连续DP与旧表示平均绝对差_km':float(np.mean(np.abs(dp[valid]-old_b2[valid])*111.2)) if valid.any() else np.nan,
        '最大多交点数':int(counts.max()),
    }
    return metrics,{'grid':grid,'old':old_b2,'first':first,'last':last,'dp':dp,'counts':counts,'spans':spans,'nearest':nearest,'dp_arc_s':ps}

# 本单元无需真实用户数据的几何单元测试
sx=np.linspace(121,146,400);sy=24+2*np.sin((sx-121)/25*5)
sg=np.array([122.,125.,130.,140.,145.])
xs=all_crossings(sx,sy,sg)
assert all(len(z)==1 for z in xs)
val,_,_=continuous_arc_selector(xs,sg)
assert np.all(np.isfinite(val))
assert np.max(np.abs(val-np.array([a[0,0] for a in xs])))<1e-8
mx=np.array([121.,124.,127.,125.,123.5,126.,130.,146.]);my=np.array([22.,24.,26.,29.,30.,31.,33.,36.])
assert max(len(x) for x in all_crossings(mx,my,np.arange(123,129,0.5)))>=2
print('[R55 GEOMETRY UNIT TEST] PASS: unique / multiple crossings, forward arclength DP')

In [ ]:
# ===== 2. 固定扫描协议 + 断点续跑（不会修改原冻结数据） =====
files=sorted(TRACKS.glob('*.npz'))
if cfg.FULL_SCAN:chosen=files
else:
    chosen=[]
    for year in range(min(dates.year),max(dates.year)+1):
        fs=[p for p in files if p.stem[:4]==str(year)]
        if not fs:continue
        if year in cfg.FOCUS_YEARS:chosen.extend(fs)
        else:
            idx=np.unique(np.linspace(0,len(fs)-1,min(cfg.SAMPLES_PER_OTHER_YEAR,len(fs)),dtype=int))
            chosen.extend([fs[j] for j in idx])
chosen=sorted(set(chosen))
print('Selected',len(chosen),'/',len(files),'tracks, full_scan=',cfg.FULL_SCAN)
RAW=DATA/'R55_01_逐日几何保真审计.csv'
existing={}
if cfg.RESUME and RAW.exists():
    old=pd.read_csv(RAW,encoding='utf-8-sig')
    if {'日期','第一分支多交点比例_复核','第一分支旧表示距真实交点P95_km'}.issubset(old.columns):
        existing=dict(zip(old['日期'].astype(str),old.to_dict(orient='records')))
    else:
        print('[R55] 旧断点缓存缺少当前第一分支审计字段，将安全重新计算；不触碰冻结数据。')
print('Already audited',len(existing))
records=list(existing.values())
errors=[]
for ix,path in enumerate(chosen,1):
    dt=path.stem; iso=f'{dt[:4]}-{dt[4:6]}-{dt[6:8]}'
    if iso in existing:continue
    try:
        obs_i=DATE_ROW[pd.Timestamp(iso)]
        with np.load(path,allow_pickle=False) as z:
            lon=np.asarray(z['lon'],float);lat=np.asarray(z['lat'],float)
        m,_=grid_reconstruction_scores(lon,lat,GRID,FROZEN_B2[obs_i]);
        m.update(branch1_scores(lon,lat,GRID_B1,FROZEN_B1[obs_i]))
        m.update({'日期':iso,'年份':int(dt[:4]),'原始点数':int(np.isfinite(lon).sum()),'状态':'OK'})
    except Exception as e:
        m={'日期':iso,'年份':int(dt[:4]),'状态':'ERROR','异常':repr(e)}
        errors.append((iso,repr(e)))
    records.append(m)
    if ix % cfg.SAVE_EVERY == 0:
        pd.DataFrame(records).sort_values('日期').to_csv(RAW,index=False,encoding='utf-8-sig')
        print(f'[scan] {ix}/{len(chosen)} saved | failures={len(errors)}')
daily=pd.DataFrame(records).drop_duplicates('日期',keep='last').sort_values('日期')
daily.to_csv(RAW,index=False,encoding='utf-8-sig')
valid=daily[daily['状态']=='OK'].copy()
if len(valid)==0:raise RuntimeError('全部几何审计失败，已保存异常日志；请检查轨迹 NPZ 键名 lon/lat。')
cols=['第一分支多交点比例_复核','第一分支旧表示距真实交点P95_km','多交点比例','无交点比例','多交点平均跨度_km','旧表示距任一真实交点平均_km','旧表示距任一真实交点P95_km','旧表示偏离真实交点超过5km比例','连续弧长DP逐格纬度跳跃P95_km','旧表示沿经度逐格纬度跳跃P95_km']
ann=valid.groupby('年份',as_index=False)[cols].mean()
ann['审计条数']=valid.groupby('年份').size().values
ann.to_csv(DATA/'R55_02_年度几何保真汇总.csv',index=False,encoding='utf-8-sig')
print('Valid days',len(valid),'errors',len(errors),'mean multi fraction',round(valid['多交点比例'].mean(),4), 'offaxis P95 mean km',round(valid['旧表示距任一真实交点P95_km'].mean(),3))
display(ann.tail(8))

In [ ]:
# ===== 3. 关键日期：输出旧表示/首次交点/末次交点/连续DP候选（只写新目录） =====
# 仅输出代表性日，不制造“全时段替换训练集”；不能用未来真实目标或模型误差来挑选策略。
rep_dates=[]
for y in cfg.FOCUS_YEARS:
    v=valid[valid['年份']==y].sort_values('旧表示距任一真实交点P95_km',ascending=False)
    rep_dates += v['日期'].head(4).tolist()
rep_dates=list(dict.fromkeys(rep_dates))
REPR=OUT/'候选表示_仅供审计';REPR.mkdir(parents=True,exist_ok=True)
example=[]
for iso in rep_dates:
    p=TRACKS/(iso.replace('-','')+'.npz')
    with np.load(p,allow_pickle=False) as z:lo=np.asarray(z['lon'],float);la=np.asarray(z['lat'],float)
    _,obj=grid_reconstruction_scores(lo,la,GRID,FROZEN_B2[DATE_ROW[pd.Timestamp(iso)]])
    np.savez_compressed(REPR/(iso.replace('-','')+'.npz'), **obj, raw_lon=lo,raw_lat=la)
    example.append({'日期':iso,'候选文件':str((REPR/(iso.replace('-','')+'.npz')).relative_to(OUT))})
pd.DataFrame(example).to_csv(DATA/'R55_03_代表性案例候选清单.csv',index=False,encoding='utf-8-sig')
print('Representative cases',len(example),'saved to',REPR)

In [ ]:
# ===== 4. R5.3 成对预测误差与 R5.5 几何指标做描述性关联 =====
# R54 ZIP 包已包含同一目标时间、同一种子的预测误差差值。
# 相关性不是因果归因，也不用于挑选修复算法。
R54=ROOT/cfg.R54_ZIP
joined=pd.DataFrame();associate=pd.DataFrame()
if R54.exists():
    with zipfile.ZipFile(R54) as z:
        paths=[n for n in z.namelist() if n.endswith('03_同一事件配对模型误差差.csv')]
        if paths:
            paired=pd.read_csv(io.BytesIO(z.read(paths[0])),encoding='utf-8-sig')
            # 每个 target-date, lead, year, seed 的成对 MSE 差值（不是两个RMSE直接相减）
            paired['目标日期']=pd.to_datetime(paired['目标日期']).dt.strftime('%Y-%m-%d')
            joined=paired.merge(valid,left_on='目标日期',right_on='日期',how='inner')
            joined.to_csv(DATA/'R55_04_几何指标与R53成对误差对应.csv',index=False,encoding='utf-8-sig')
            features=['多交点比例','多交点平均跨度_km','旧表示距任一真实交点P95_km','旧表示偏离真实交点超过5km比例']
            results=[]
            for yr,lead in sorted(set(zip(joined['QA年份'],joined['提前期_天']))):
                block=joined[(joined['QA年份']==yr)&(joined['提前期_天']==lead)].copy()
                # seed不能算独立样本：按目标日期聚合每个日期的种子均值
                b=block.groupby('目标日期',as_index=False).mean(numeric_only=True)
                yy='PCConcat减Matched-LSTM_MSE_km2'
                for key in features:
                    v=b[[key,yy]].dropna()
                    r=float(v[key].corr(v[yy],method='spearman')) if len(v)>10 and v[key].nunique()>2 else np.nan
                    results.append({'QA年份':yr,'提前期_天':lead,'几何特征':key,'Spearman描述性相关':r,'独立目标日数':len(v)})
            associate=pd.DataFrame(results)
            associate.to_csv(DATA/'R55_05_按年份与提前期的描述性关联.csv',index=False,encoding='utf-8-sig')
            print('Paired observations matched:',len(joined),'distinct target days:',joined['目标日期'].nunique())
        else:print('[WARN] R54 ZIP lacks paired prediction table; geometry-only results are intact.')
else:print('[WARN] R54 ZIP absent; geometry-only results are intact.')
if len(associate):display(associate)

In [ ]:
# ===== 5. 保持中英双份图；中文字体不可用时跳过中文图，避免乱码 =====
zh_candidates=['/mnt/c/Windows/Fonts/msyh.ttc','/mnt/c/Windows/Fonts/simhei.ttf','/mnt/c/Windows/Fonts/simsun.ttc','/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc']
CH_FONT=None
for f in zh_candidates:
    if Path(f).exists():
        try:
            font_manager.fontManager.addfont(f)
            CH_FONT=font_manager.FontProperties(fname=f).get_name();break
        except Exception:pass
if CH_FONT is None:warnings.warn('未找到中文字体：不绘制乱码中文图；英文图正常输出。建议安装 fonts-noto-cjk')

def draw_bilingual(stem,fn):
    for lang,folder in [('zh',IM_ZH),('en',IM_EN)]:
        if lang=='zh' and CH_FONT is None:continue
        matplotlib.rcParams.update({'font.family':[CH_FONT,'DejaVu Sans'] if lang=='zh' else ['DejaVu Serif'],
            'axes.unicode_minus':False,'pdf.fonttype':42,'axes.grid':True,'grid.alpha':0.2,
            'xtick.direction':'in','ytick.direction':'in','figure.facecolor':'white','axes.facecolor':'white'})
        fig=fn(lang)
        fig.savefig(folder/f'{stem}_{"中文" if lang=="zh" else "English"}.png',dpi=cfg.FIG_DPI,bbox_inches='tight')
        fig.savefig(folder/f'{stem}_{"中文" if lang=="zh" else "English"}.pdf',bbox_inches='tight')
        plt.close(fig)

def fig_annual(lang):
    fig,ax=plt.subplots(figsize=(9.0,4.2))
    ax.plot(ann['年份'],100*ann['多交点比例'],'-o',markersize=3,color='#C88463',label='第二分支' if lang=='zh' else 'Branch 2')
    ax.plot(ann['年份'],100*ann['第一分支多交点比例_复核'],'-s',markersize=3,color='#4F7594',label='第一分支' if lang=='zh' else 'Branch 1')
    ax.legend(fontsize=8)
    ax.set_xlabel('年份' if lang=='zh' else 'Year');ax.set_ylabel('多交点经线采样比例 (%)' if lang=='zh' else 'Multiple-crossing grid (%)')
    ax.set_title('R5.5 第二分支经线多交点审计（分层抽样）' if lang=='zh' else 'R5.5 Branch-2 multiple crossings (stratified sample)')
    fig.tight_layout();return fig

def fig_offaxis(lang):
    fig,ax=plt.subplots(figsize=(9.0,4.2))
    ax.plot(ann['年份'],ann['旧表示距任一真实交点P95_km'],'-o',markersize=3,color='#4F7594')
    ax.set_xlabel('年份' if lang=='zh' else 'Year');ax.set_ylabel('旧表示距任一真实交点 P95（km）' if lang=='zh' else 'Old representation to any crossing P95 (km)')
    ax.set_title('单值插值的真实轴线偏离风险' if lang=='zh' else 'Distance of single-valued interpolation from track intersections')
    fig.tight_layout();return fig

draw_bilingual('图1_逐年多交点比例',fig_annual)
draw_bilingual('图2_旧表示距真实交点偏差',fig_offaxis)

if example:
    selected=example[0]['日期'];p=REPR/(selected.replace('-','')+'.npz')
    with np.load(p,allow_pickle=False) as z:
        rawlon=z['raw_lon'];rawlat=z['raw_lat'];b2old=z['old'];b2dp=z['dp'];b2first=z['first'];gg=z['grid']
    def fig_example(lang):
        fig,ax=plt.subplots(figsize=(9,4.8))
        ax.plot(rawlon,rawlat,lw=1.7,color='#345A77',label='冻结原始流轴' if lang=='zh' else 'Frozen raw axis')
        ax.plot(gg,b2old,lw=1.0,alpha=.8,color='#C88463',label='原双分支表示' if lang=='zh' else 'Original branch-2')
        ax.plot(gg,b2dp,'-',lw=1.0,alpha=.85,color='#53A397',label='连续弧长DP交点' if lang=='zh' else 'Monotone-arclength DP')
        ax.set_xlim(121.5,145.5);ax.set_xlabel('经度（°E）' if lang=='zh' else 'Longitude (°E)');ax.set_ylabel('纬度（°N）' if lang=='zh' else 'Latitude (°N)')
        ax.set_title(('路径表示案例 '+selected) if lang=='zh' else ('Representation example '+selected));ax.legend(fontsize=8)
        fig.tight_layout();return fig
    draw_bilingual('图3_真实轨迹与候选单值分支',fig_example)
print('Images ready; Chinese font=',CH_FONT)

In [ ]:
# ===== 6. 准入诊断、结果清单、自动 ZIP + SHA256 + 文件链接 =====
# Gate仅是“是否值得开展独立的表示修复小试验”，绝不等于已经改进预测。
focus=valid[valid['年份'].isin(cfg.FOCUS_YEARS)]
if focus.empty:focus=valid
stats={
    'processed_valid_tracks':int(len(valid)),
    'processed_failures':int((daily['状态']=='ERROR').sum()),
    'total_frozen_tracks':len(files),
    'geometry_sampled_fraction':float(len(valid)/max(1,len(files))),
    'mean_multi_cross_grid_share':float(valid['多交点比例'].mean()),
    'branch1_mean_multi_cross_grid_share':float(valid['第一分支多交点比例_复核'].mean()),
    'branch1_mean_offaxis_P95_km':float(valid['第一分支旧表示距真实交点P95_km'].mean()),
    'focus_mean_multi_cross_grid_share':float(focus['多交点比例'].mean()),
    'focus_mean_multicross_spread_km':float(focus['多交点平均跨度_km'].mean()),
    'focus_mean_offaxis_km':float(focus['旧表示距任一真实交点平均_km'].mean()),
    'focus_mean_offaxis_P95_km':float(focus['旧表示距任一真实交点P95_km'].mean()),
    'focus_mean_offaxis_fraction_gt5km':float(focus['旧表示偏离真实交点超过5km比例'].mean()),
    'R53_model_qualification':'failed (source R5.3/R5.4)',
    'R55_prediction_improvement_established':False,
    'R55_dataset_modified':False,
    'known_limitations':[
      'No independent observation-based axis truth; frozen tracks are a reference, not physical ground truth.',
      'A single-valued longitude→latitude mapping cannot preserve all loops/multiple crossings.',
      'DP is a geometry-only candidate; being on the track does not guarantee choosing the physically dominant branch.',
      'QA years 2008/2016/2012/2020 are already exposed in model development and cannot be blind tests.'
    ],
}
# 仅在风险非微小且有物理多值性时建议推进表示修复的机制小实验；不建议开展正式长训
meaningful=(stats['focus_mean_multicross_spread_km']>=15.0 and
            (stats['focus_mean_offaxis_P95_km']>=5.0 or stats['focus_mean_offaxis_fraction_gt5km']>=0.05))
stats['geometry_mechanism_pilot_recommended']=bool(meaningful)
stats['formal_training_qualified']=False
(REPORT/'R55_最终诊断交接.json').write_text(json.dumps(stats,ensure_ascii=False,indent=2),encoding='utf-8')
lines=['# R5.5 路径表示审计', '',f"- 受检有效流轴：{stats['processed_valid_tracks']} / {stats['total_frozen_tracks']}",
       f"- 第二分支多交点经线平均比例：{stats['mean_multi_cross_grid_share']:.2%}",
       f"- 第一分支多交点纬线平均比例：{stats['branch1_mean_multi_cross_grid_share']:.2%}",
       f"- 研发关注年份：多交点分支经线纬度跨度均值：{stats['focus_mean_multicross_spread_km']:.2f} km",
       f"- 研发关注年份：原单值编码距交点的 P95 日平均值：{stats['focus_mean_offaxis_P95_km']:.2f} km",
       f"- 是否值得继续做**独立表示机制小试验**：{meaningful}（并非长训资格）",'',
       '- 注意：真实蛇行≠错误流轴；可修复的偏离≠不可恢复的多值信息损失。',
       '- 2008/2016及2012/2020年已经用于研发，不允许声称为真正盲测。',
       '- 本轮只做冻结数据诊断，不改文件，不重训，不剔除日期。']
(REPORT/'R55_结论与下一步.md').write_text('\n'.join(lines),encoding='utf-8')

archive=ROOT/'R55_上传给ChatGPT_几何保真审计.zip'
sha=Path(str(archive)+'.sha256')
files_out=sorted(p for p in OUT.rglob('*') if p.is_file() and p.stat().st_size<=cfg.MAX_ARCHIVE_MB*1024**2
                 and p.name!='R55_文件清单_SHA256.csv')
manifest=[]
for p in files_out:
    manifest.append({'相对路径':str(p.relative_to(OUT)),'大小字节':p.stat().st_size,'SHA256':hashlib.sha256(p.read_bytes()).hexdigest()})
pd.DataFrame(manifest).to_csv(REPORT/'R55_文件清单_SHA256.csv',index=False,encoding='utf-8-sig')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=5,allowZip64=True) as z:
    for p in files_out:z.write(p,arcname='R55_结果/'+p.relative_to(OUT).as_posix())
    z.write(REPORT/'R55_文件清单_SHA256.csv',arcname='R55_结果/报告/R55_文件清单_SHA256.csv')
with zipfile.ZipFile(archive) as z:
    corrupt=z.testzip()
    if corrupt:raise RuntimeError(f'ZIP内部校验失败: {corrupt}')
shaval=hashlib.sha256(archive.read_bytes()).hexdigest()
sha.write_text(f'{shaval}  {archive.name}\n',encoding='utf-8')
print('[R55 ARCHIVE PASS]',archive,'bytes=',archive.stat().st_size,'SHA256=',shaval)
display(Markdown('**R5.5 全部结果已打包：**'))
display(FileLink(str(archive)))
print('Windows 资源管理器：',str(archive).replace('/mnt/d/','D:\\').replace('/','\\'))
print('正式训练资格：NO；下一轮是否值得做表示机制小试验：',meaningful)